<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Tag your documents when you upload them, then ask by tag.**

Tags carry what a file does not say about itself: which team owns it, what kind of document it is, which year it belongs to. This notebook demonstrates how to attach **your own metadata** to documents on PageIndex Cloud and use it in chat.

We upload four documents, each tagged with its `type`, `team`, and `year`. The tags come back with every document, and the chat agent sees them when it looks through your library, so a question can name documents by their tags instead of their file names.

1. **Tag on upload** — Pass a `metadata` dict to `submit_document()`.
2. **Read the tags back** — Get them from `get_document()`.
3. **Ask by tag** — Name documents by their tags in a chat question.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) and OpenAI API key ready. This demo uploads four small PDFs to your PageIndex Cloud library.


## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the demo documents.


In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your API keys and choose your chat `MODEL`. The client indexes documents on PageIndex Cloud, and your chosen model answers with PageIndex Cloud's retrieval tools.


In [2]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

client = PageIndexClient(
    index="cloud",
    chat=MODEL,
)

## 02 · Tag documents on upload

Download four PDFs from the PageIndex repository and submit each one with its tags. `team` is something no file states about itself; only the tag knows it. `wait=True` returns once the document is ready. A document already in your library from an earlier run is reused rather than uploaded again.


In [3]:
import requests
from pageindex import PageIndexAPIError

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "q1-fy25-earnings.pdf": {"type": "earnings_release", "team": "finance", "year": 2025},
    "Regulation Best Interest_Interpretive release.pdf": {"type": "regulation", "team": "legal", "year": 2019},
    "earthmover.pdf": {"type": "paper", "team": "research", "year": 2013},
    "attention-residuals.pdf": {"type": "paper", "team": "research", "year": 2026},
}

doc_ids = {}
for file_name, metadata in DOCS.items():
    try:
        doc_ids[file_name] = client.get_document_id(file_name)  # already in your library
        print(f"{file_name}: {doc_ids[file_name]} (already uploaded)")
        continue
    except PageIndexAPIError:
        pass

    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(file_name, wait=True, metadata=metadata)["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

q1-fy25-earnings.pdf: pi-cmvhzgweq00000bp9a43qzish
Regulation Best Interest_Interpretive release.pdf: pi-cmvhziee400010bp990t2z89i
earthmover.pdf: pi-cmvhzjwos00000cpbeuapn4ya
attention-residuals.pdf: pi-cmvhzkhfe00010cpbl79pv7bn


## 03 · Read the tags back

The tags come back under `metadata` from `get_document()`, and on every document from `list_documents()`. Metadata is set at submission; to change it, delete the document and submit it again.


In [4]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

q1-fy25-earnings.pdf: {'team': 'finance', 'type': 'earnings_release', 'year': 2025}
Regulation Best Interest_Interpretive release.pdf: {'team': 'legal', 'type': 'regulation', 'year': 2019}
earthmover.pdf: {'team': 'research', 'type': 'paper', 'year': 2013}
attention-residuals.pdf: {'team': 'research', 'type': 'paper', 'year': 2026}


## 04 · Ask by tag

Ask about "the finance team's earnings release" without naming the file. The agent finds the document by its tags, then reads its pages. The output streams the answer alongside retrieval tool calls.


In [5]:
messages = [{"role": "user", "content": "What revenue did the finance team's 2025 earnings release report?"}]

response = None
for event in client.chat(messages, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] get_folder_structure({})

[Tool call] browse_documents({"folder_id":"root","recursive":false,"sort":"relevance","query":"finance team 2025 earnings release revenue","limit":10})

[Tool call] get_document({"doc_name":"q1-fy25-earnings.pdf","wait_for_completion":true})

[Tool call] get_document_structure({"doc_name":"q1-fy25-earnings.pdf","part":1,"wait_for_completion":true})

[Tool call] get_page_content({"doc_name":"q1-fy25-earnings.pdf","pages":"1-4","wait_for_completion":true})
The finance team’s 2025 Q1 earnings release reported **$24.690 billion in revenue** for the quarter ended December 28, 2024—up **5%** from $23.549 billion in Q1 fiscal 2024.

Tags also answer questions on their own. List the research team's papers: the agent reads the team and year from the tags, not from the papers.


In [6]:
messages = [{"role": "user", "content": "Which papers does the research team have? List each with its year, oldest first."}]

response = None
for event in client.chat(messages, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] get_folder_structure({})

[Tool call] browse_documents({"folder_id":"root","recursive":false,"sort":"relevance","query":"research team papers","limit":50})

[Tool call] get_metadata_schema({})
The research team has these papers, oldest first:

1. **earthmover.pdf** — **2013**
2. **attention-residuals.pdf** — **2026**

Metadata is a flat dict of `str`, `int`, `float`, or `bool` values (numbers must be finite, so no `NaN` or `Infinity`), with keys that start with a letter and contain only letters, digits, and underscores. Its serialized size is at most 8 KB, and PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See the [metadata documentation](https://docs.pageindex.ai/sdk/documents#metadata) for details.


## 05 · Review usage & estimated cost (optional)

Inspect the token usage for the last answer and estimate its model cost using the rate table below. Tagging documents and reading the tags back do not require model calls. This estimate does not include PageIndex Cloud charges.


In [7]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 25466,
  "input_tokens_details": {
    "cached_tokens": 18019,
    "cache_write_tokens": 7435
  },
  "output_tokens": 405,
  "output_tokens_details": {
    "reasoning_tokens": 287
  },
  "total_tokens": 25871,
  "request_usage_entries": [
    {
      "input_tokens": 5340,
      "input_tokens_details": {
        "cached_tokens": 0,
        "cache_write_tokens": 5337
      },
      "output_tokens": 35,
      "output_tokens_details": {
        "reasoning_tokens": 18
      },
      "total_tokens": 5375
    },
    {
      "input_tokens": 5580,
      "input_tokens_details": {
        "cached_tokens": 5337,
        "cache_write_tokens": 240
      },
      "output_tokens": 54,
      "output_tokens_details": {
        "reasoning_tokens": 13
      },
      "total_tokens": 5634
    },
    {
      "input_tokens": 7108,
      "input_tokens_details": {
        "cached_tokens": 5577,
        "cache_write_tokens": 1528
      },
      "output_tokens": 88,
      "output_toke

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
